# SplatAD / OmniRE trên PandaSet

Đọc PandaSet từ Google Drive đã mount, train trên Colab GPU và phát lại **toàn bộ scene** theo ba camera trước cùng timeline. Dashboard có chọn scene, chọn cấu hình, đổi giữa ba góc và từng camera, cùng mục Hiệu chỉnh SMPL.

- Mặc định dùng `STAGE="curriculum"` và R2. Thứ tự là `028` (gần) → `011` (nhiều người) → `008` (xa và bị che). Mỗi cảnh train một mô hình riêng với ba camera trước và xuất MP4 60 FPS.
- Chọn `STAGE="scene"` cho một cảnh hoặc đặt danh sách `SCENES` để tiếp tục từ cảnh mong muốn.
- Đặt `PANDASET_PATH` nếu cần, trỏ tới thư mục đã giải nén chứa các folder `001`, `011`, … hoặc ZIP PandaSet đang lưu trên Drive.
- R1/R2 dùng SMPL Neutral. Nếu đã có file trên Drive, đặt `SMPL_PICKLE` và `MODEL_VERSION` theo file. Chế độ R0 chỉ cần PandaSet.

Các clip camera gốc được tạo ở cell xem trước từ Drive. Các clip tái tạo được xuất sau khi mỗi mô hình có checkpoint. [PandaSet](https://huggingface.co/datasets/georghess/pandaset) · [SMPL](https://smpl.is.tue.mpg.de/) · [Pose OmniRe](https://github.com/ziyc/drivestudio/blob/e59bda4fa681f829dbb1d65f0de582b0f633c450/docs/HumanPose.md)

In [ ]:
from collections import Counter
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time

from google.colab import drive, output

DRIVE = Path("/content/drive")
drive.mount(str(DRIVE))
PERSIST = DRIVE / "MyDrive/SplatAD-OmniRE"
PERSIST.mkdir(exist_ok=True)
REPO = Path("/content/SplatAD-OmniRE")
GITHUB = "https://github.com/buide03/SplatAD-OmniRE.git"
ARCHIVE = PERSIST / "SplatAD-OmniRE.zip"

def shell(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

if not (REPO / "versions.json").exists():
    if ARCHIVE.is_file():
        REPO.mkdir(exist_ok=True)
        shutil.unpack_archive(ARCHIVE, REPO)
    else:
        shell("git", "clone", GITHUB, REPO)
curriculum_path = REPO / "protocol/curriculum.json"
if not curriculum_path.is_file() and (REPO / ".git").is_dir():
    shell("git", "pull", "--ff-only", cwd=REPO)
assert curriculum_path.is_file(), "Cần bản mã nguồn mới có lộ trình train theo độ khó."

sys.path.insert(0, str(REPO))
from research.pandaset_input import prepare_source, select_source
from research.sync_video import FRONT_CAMERAS

PANDASET_PATH = ""  # Ví dụ /content/drive/MyDrive/datasets/pandaset hoặc /content/drive/MyDrive/pandaset.zip
STAGE = "curriculum"  # curriculum | scene | smoke | pilot | confirm
RUN_ID = f"{STAGE}-front3"
VARIANTS = ("R2",) if STAGE in ("curriculum", "scene") else ("R0", "R1", "R2")
STEPS = {"curriculum": 1200, "scene": 1200, "smoke": 200, "pilot": 1200, "confirm": 4000}[STAGE]
CURRICULUM = {entry["scene_id"]: entry for entry in json.loads(curriculum_path.read_text(encoding="utf-8"))["scenes"]}
PILOT = json.loads((REPO / "protocol/pilot.json").read_text(encoding="utf-8"))
if STAGE == "curriculum":
    SCENES = list(CURRICULUM)
elif STAGE in ("scene", "smoke"):
    SCENES = ["011"]
else:
    SCENES = [entry["scene_id"] for entry in PILOT["scenes"]]
TRAIN_CAMERAS = [*FRONT_CAMERAS, "left", "right", "back"] if STAGE in ("pilot", "confirm") else list(FRONT_CAMERAS)
CAMERAS = TRAIN_CAMERAS
REPLAY_CAMERAS = FRONT_CAMERAS
REPLAY_FPS = 60
REPLAY_SCALE = 1.0  # Giữ độ phân giải gốc cho từng camera
USE_PUBLIC_POSES = True
NEED_POSE = any(variant != "R0" for variant in VARIANTS)
SMPL_PICKLE = PERSIST / "SMPL_NEUTRAL.pkl"
MODEL_VERSION = "smpl_neutral_v1.1"
if STAGE == "curriculum" and NEED_POSE:
    RUN_ID = f"{RUN_ID}-{MODEL_VERSION}"
PROCESSED = PERSIST / ("omnire-pose" if USE_PUBLIC_POSES else "omnire-pose-local")
INPUTS = PERSIST / "inputs"
RUNS = PERSIST / "runs"
ARTIFACTS = PERSIST / "artifacts"
RESULTS = ARTIFACTS / RUN_ID
tag_cameras = "front3" if tuple(CAMERAS) == FRONT_CAMERAS else "surround"
tag_pose = ("public-pose" if USE_PUBLIC_POSES else "local-pose") if NEED_POSE else "rigid"
PREP_TAG = f"{tag_cameras}-{tag_pose}"
if STAGE == "curriculum":
    PREP_TAG = f"curriculum-{PREP_TAG}"
    if NEED_POSE:
        PREP_TAG = f"{PREP_TAG}-{MODEL_VERSION}"
for folder in (INPUTS, RUNS, ARTIFACTS, RESULTS):
    folder.mkdir(exist_ok=True)

read_cameras = tuple(CAMERAS) if USE_PUBLIC_POSES or not NEED_POSE else (*FRONT_CAMERAS, "left", "right", "back")
SOURCE, SCENES = select_source(PANDASET_PATH or None, [DRIVE / "MyDrive", DRIVE / "Shareddrives"], SCENES)

def raw_for(scene):
    return prepare_source(SOURCE, Path("/content/pandaset-from-drive"), [scene], read_cameras)

print(f"PandaSet: {SOURCE}")
print("Thứ tự train", " → ".join(SCENES))

## Xem ba camera gốc và monitor

Cell này tạo preview đồng bộ từ Drive cho scene đầu tiên và mở dashboard. Chọn **Ba camera trước** để phát hoặc tua cùng lúc ba góc. Mục **Hiệu chỉnh SMPL** sẽ tự cập nhật khi R1 hoặc R2 chạy.

In [ ]:
if shutil.which("ffmpeg") is None:
    shell("apt-get", "-qq", "update")
    shell("apt-get", "-qq", "install", "-y", "ffmpeg")

preview_scene = SCENES[0]
source_folder = ARTIFACTS / "source" / preview_scene
if not (source_folder / "run.json").exists() or not (source_folder / "front-triptych.mp4").exists():
    shell(sys.executable, "-m", "research.source_preview", "--raw", raw_for(preview_scene), "--scene", preview_scene,
          "--front-triptych", "--output", source_folder, cwd=REPO)

if "dashboard" in globals() and dashboard.poll() is None:
    dashboard.terminate()
    dashboard.wait(timeout=5)
dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                              "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
time.sleep(1)
assert dashboard.poll() is None, "Không khởi động được dashboard."
output.serve_kernel_port_as_iframe(8765, height=1000)

## Môi trường train

Chọn GPU trong Colab trước khi chạy cell này. Bootstrap dùng Python 3.12, PyTorch CUDA 12.8 hoặc 12.6 và fork GSplat theo commit đã pin. Sau khi nạp checkpoint, repo render ba camera theo cùng mốc thời gian rồi ghép thành một clip.

In [ ]:
shell("nvidia-smi")
shell("bash", "scripts/bootstrap-colab.sh", "train", cwd=REPO)
ACTIVATE = REPO / ".deps/train-env/activate-research"

def research(module, *args):
    shell("bash", "-c", 'source "$1"; shift; exec python -m "$@"',
          "splatad", ACTIVATE, f"research.{module}", *args, cwd=REPO)

## Pose khởi tạo và manifest

R1 và R2 dùng SMPL. `USE_PUBLIC_POSES=True` nạp pose do OmniRe công bố. Chọn `False` để chạy 4D-Humans cho cảnh chưa có pose. Notebook chuẩn bị pose và manifest theo từng cảnh trước khi train. Replay bao phủ toàn bộ khoảng thời gian của cảnh. Để chạy R0 trên toàn bộ dữ liệu, đặt `STAGE="scene"`, `VARIANTS=("R0",)` và `SCENES="all"` ở cell cấu hình.

In [ ]:
BODY = INPUTS / f"{MODEL_VERSION}.npz"
if NEED_POSE:
    if not BODY.exists():
        assert SMPL_PICKLE.is_file(), f"Đặt mô hình Neutral hợp lệ tại {SMPL_PICKLE}"
        SMPL_ENV = REPO / ".deps/smpl-convert"
        shell("uv", "venv", "--allow-existing", "--python", "3.10", SMPL_ENV, cwd=REPO)
        smpl_python = SMPL_ENV / "bin/python"
        shell("uv", "pip", "install", "--python", smpl_python, "setuptools<70", "wheel", "pip", cwd=REPO)
        shell("uv", "pip", "install", "--python", smpl_python, "--no-build-isolation",
              "numpy==1.26.4", "scipy==1.12.0", "chumpy==0.70", cwd=REPO)
        shell(smpl_python, "-m", "research.prepare", "prepare-smpl",
              "--input", SMPL_PICKLE, "--output", BODY, "--model-version", MODEL_VERSION, cwd=REPO)

    if not USE_PUBLIC_POSES:
        assert SMPL_PICKLE.is_file(), f"Đặt mô hình Neutral hợp lệ tại {SMPL_PICKLE}"
        shell("bash", "scripts/bootstrap-colab.sh", "pose", cwd=REPO)

def tracks_for(scene):
    return INPUTS / f"{scene}-{PREP_TAG}-tracks.json"

def manifest_for(scene):
    return INPUTS / f"{scene}-{PREP_TAG}-manifest.json"

def check_curriculum(scene, spec):
    stage = CURRICULUM[scene]
    selected = set(spec["selected_uuids"])
    moving = [row for row in spec["observations"]
              if row["uuid"] in selected and row["smpl_valid"] and not row["stationary"]]
    near = {row["uuid"] for row in moving if row["bin"] == "near" and row["usable"]}
    per_frame = Counter((row["camera"], row["frame"]) for row in moving if row["usable"])
    simultaneous = max(per_frame.values(), default=0)
    far_occluded = {row["uuid"] for row in moving if row["bin"] == "far"
                    and row["limbs_visible"] is False and row["pose_monitor"]}
    print(f"{scene} · {stage['label']} · {len(near)} track gần · đỉnh {simultaneous} người cùng khung · "
          f"{len(far_occluded)} track xa có mốc bị che để monitor")
    qualifies = {"near": bool(near), "multiple": simultaneous >= 3, "far_occluded": bool(far_occluded)}
    if not qualifies[stage["focus"]]:
        raise ValueError(f"Manifest {scene} thiếu quan sát SMPL cho trọng tâm {stage['label']}")

def prepare_scene(scene, raw):
    tracks, manifest = tracks_for(scene), manifest_for(scene)
    if NEED_POSE and not tracks.exists():
        if USE_PUBLIC_POSES:
            research("omnire", "--raw", raw, "--processed", PROCESSED,
                     "--archive", INPUTS / "pandaset_processed_humanpose.zip", "--scenes", scene)
        elif not (PROCESSED / scene / "humanpose/smpl.pkl").exists():
            shell("bash", "scripts/omnire-pose.sh", raw, PROCESSED, scene, SMPL_PICKLE, cwd=REPO)
        research("prepare", "export-tracks", "--raw", raw, "--processed", PROCESSED,
                 "--body", BODY, "--scene", scene, "--output", tracks)
    if not manifest.exists():
        review = INPUTS / f"{scene}-review.json"
        if not review.is_file():
            review = REPO / "protocol" / f"{scene}-review.json"
        research("prepare", "inventory", "--raw", raw, "--scene", scene,
                 "--cameras", *CAMERAS, "--output", manifest,
                 *(["--tracks", tracks] if NEED_POSE else []),
                 *(["--review", review] if review.is_file() else []))
    if NEED_POSE:
        spec = json.loads(manifest.read_text(encoding="utf-8"))
        print(scene, len(spec["selected_uuids"]), "UUID định tuyến SMPL")
        assert spec["selected_uuids"], f"Cần track SMPL hợp lệ cho cảnh {scene}"
        if STAGE == "curriculum" and scene in CURRICULUM:
            check_curriculum(scene, spec)
        if STAGE in ("pilot", "confirm"):
            simultaneous = {}
            for row in spec["observations"]:
                if row["usable"] and row["smpl_valid"] and row["uuid"] in spec["selected_uuids"]:
                    simultaneous.setdefault((row["camera"], row["frame"]), set()).add(row["uuid"])
            assert max(map(len, simultaneous.values()), default=0) >= 2, f"Cảnh {scene} cần hai UUID SMPL cùng xuất hiện"

## Train và xuất toàn cảnh theo từng checkpoint

Notebook lần lượt chuẩn bị, train và xuất ba clip tái tạo cùng clip ghép cho từng cảnh. Mỗi mô hình học toàn cảnh với checkpoint riêng. R2 ghi tiến trình hiệu chỉnh SMPL lên dashboard trong lúc train. Khi chạy lại, notebook tiếp tục từ checkpoint của từng cảnh.

In [ ]:
def run_folder(scene, variant):
    return RUNS / f"{scene}-{variant}" / "splatad-smpl" / RUN_ID

def checkpoint_for(scene, variant):
    folder = run_folder(scene, variant)
    return max((folder / "nerfstudio_models").glob("step-*.ckpt"), default=None)

def replay_args(scene, variant, zero_velocity=False):
    checkpoint = checkpoint_for(scene, variant)
    suffix = "-velocity-off" if zero_velocity else ""
    folder = RESULTS / f"{scene}-{variant}{suffix}-{checkpoint.stem}-{REPLAY_FPS}fps-{REPLAY_SCALE:g}x"
    shared = ["--config", run_folder(scene, variant) / "config.yml", "--checkpoint", checkpoint,
              "--manifest", manifest_for(scene), "--output", folder]
    if zero_velocity:
        shared.append("--zero-smpl-velocity")
    return folder, shared

def render_scene(scene, variant, zero_velocity=False):
    folder, shared = replay_args(scene, variant, zero_velocity)
    artifact = folder / "run.json"
    if artifact.is_file():
        record = json.loads(artifact.read_text(encoding="utf-8"))
        triptych = record.get("front_triptych", {})
        files = [entry["file"] for entry in record.get("videos", [])]
        if (triptych.get("cameras") == list(REPLAY_CAMERAS) and len(files) == len(REPLAY_CAMERAS)
                and all((folder / file).is_file() for file in [*files, triptych["file"]])):
            print(f"Replay đã có: {folder}")
            return
    research("replay", *shared, "--cameras", *REPLAY_CAMERAS, "--fps", REPLAY_FPS, "--scale", REPLAY_SCALE)

for stage_index, scene in enumerate(SCENES, start=1):
    focus = CURRICULUM.get(scene, {}).get("label", "") if STAGE == "curriculum" else ""
    print(f"[{stage_index}/{len(SCENES)}] Cảnh {scene} · {focus or STAGE}", flush=True)
    raw = raw_for(scene)
    prepare_scene(scene, raw)
    for variant in VARIANTS:
        folder = run_folder(scene, variant)
        config = folder / "config.yml"
        live_monitor = ["--pose-monitor-dir", RESULTS / f"{scene}-{variant}-pose",
                        "--pose-monitor-every", 50 if STAGE == "smoke" else 250]
        checkpoint = checkpoint_for(scene, variant)
        done = int(checkpoint.stem.rsplit("-", 1)[1]) + 1 if checkpoint else 0
        if done < STEPS:
            if checkpoint:
                research("train", "--resume", config, "--checkpoint", checkpoint, "--steps", STEPS - done, *live_monitor)
            else:
                pose_args = ["--tracks", tracks_for(scene), "--body", BODY] if variant != "R0" else []
                research("train", "--raw", raw, "--scene", scene, "--variant", variant,
                         *pose_args, "--manifest", manifest_for(scene),
                         "--cameras", *CAMERAS, "--output", RUNS, "--run-id", RUN_ID, "--steps", STEPS,
                         "--checkpoint-every", 200 if STAGE == "smoke" else 500,
                         "--monitor-every", 200 if STAGE == "smoke" else 500, *live_monitor)
        render_scene(scene, variant)

## Metric và đối chứng

Cell này bổ sung metric cho các replay đã xuất. Đánh giá dùng ảnh và tia tại mốc cảm biến gốc. Đặt `CHECK_VELOCITY=True` để render thêm đối chứng R2/vận tốc 0. Đặt `RUN_GRADIENT_PROBES=True` để chạy probe RGB/LiDAR từ checkpoint sau khi xuất video.

In [ ]:
PROBES_ONLY = STAGE != "confirm"
CHECK_VELOCITY = STAGE in ("smoke", "pilot", "confirm")
RUN_GRADIENT_PROBES = STAGE in ("smoke", "pilot", "confirm")
folders = []
for scene in SCENES:
    raw_for(scene)
    for variant in VARIANTS:
        velocity_modes = (False, True) if CHECK_VELOCITY and variant == "R2" else (False,)
        for zero_velocity in velocity_modes:
            if zero_velocity:
                render_scene(scene, variant, zero_velocity=True)
            folder, shared = replay_args(scene, variant, zero_velocity)
            research("evaluate", *shared, *(["--probes"] if PROBES_ONLY else []))
            folders.append(folder)
if len(VARIANTS) > 1 or (CHECK_VELOCITY and "R2" in VARIANTS):
    research("compare", "--runs", *folders, "--output", RESULTS / "comparison")

if RUN_GRADIENT_PROBES:
    for scene in SCENES:
        for variant in VARIANTS:
            if variant in ("R0", "R2"):
                research("probe", "--config", run_folder(scene, variant) / "config.yml",
                         "--checkpoint", checkpoint_for(scene, variant), "--manifest", manifest_for(scene),
                         "--output", RESULTS / f"{scene}-{variant}-probe.json")

## Dashboard sau khi render

Chọn scene và cấu hình đã tái tạo. **Ba camera trước** phát clip đồng bộ, theo thứ tự trái, giữa, phải. Chọn tên camera để chuyển sang clip riêng và giữ vị trí phát trên cùng timeline.

In [ ]:
if "dashboard" not in globals() or dashboard.poll() is not None:
    dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                                  "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
    time.sleep(1)
output.serve_kernel_port_as_iframe(8765, height=1000)